# Pydantic AI search diagnostics

This notebook intentionally ignores the project prompts. It checks the real chain in three steps:

1. model returns a plain answer without tools;
2. model may call the real `search_spots` tool and returns plain text;
3. model must call the real `search_spots` tool and return a structured Pydantic object.


In [16]:
import json
import asyncio
import os
from pathlib import Path
from pprint import pprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent

from capabilities.search import SearchCapability
from capabilities import Thinking
from core.models import StargazingSpot


async def run_with_timeout(label: str, awaitable, timeout: float = 45):
    print(f"START: {label}")
    result = await asyncio.wait_for(awaitable, timeout=timeout)
    print(f"DONE: {label}")
    return result


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value


load_dotenv()

MODEL = os.environ["LAZY_STELLAR_MODEL"]

print(f"MODEL: {MODEL}")
for key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'missing'}")


ImportError: cannot import name 'Thinking' from 'capabilities' (unknown location)

In [2]:
class SpotSearchReport(BaseModel):
    spots_found: int = Field(description="Number of spots returned by the search tool.")
    summary: str = Field(description="Concise user-facing answer in English.")
    spots: list[StargazingSpot] = Field(default_factory=list)


## 1. Plain model call, no tools

If this hangs or fails, the problem is model/provider configuration, not search or structured output.


In [3]:
plain_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    instructions="Reply with one short sentence. Do not use tools.",
)

plain_result = await run_with_timeout(
    "plain model call",
    plain_agent.run("Say hello and name one reason dark skies matter."),
)

print("OUTPUT:")
print(plain_result.output)
print("\nUSAGE:")
print(plain_result.usage)


START: plain model call
DONE: plain model call
OUTPUT:
Hello, dark skies are essential because they provide crucial habitats for nocturnal wildlife.

USAGE:
RunUsage(input_tokens=21, output_tokens=15, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)


## 2. Real search tool, plain text output

If step 1 works but this fails, inspect whether the model called `search_spots`, whether Tavily/DuckDuckGo failed, or whether the tool result came back as an error string.


In [4]:
search_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. For location-specific recommendations, "
        "call search_spots exactly once before answering. Then summarize the returned spots. "
        "If the tool returns an error string, report that exact failure briefly."
    ),
)

search_text_result = await run_with_timeout(
    "search tool + plain text output",
    search_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("OUTPUT:")
print(search_text_result.output)
print("\nUSAGE:")
print(search_text_result.usage)
print("\nMESSAGES:")
for message in search_text_result.all_messages():
    print(type(message).__name__, message)


START: search tool + plain text output


/Users/johnwunderbellen/my-lazy-stellar/capabilities/search.py:27: LogfireNotConfiguredWarning: No logs or spans will be created until `logfire.configure()` has been called. Set the environment variable LOGFIRE_IGNORE_NO_CONFIG=1 or add ignore_no_config=true in pyproject.toml to suppress this warning.
  logfire.info("Using Tavily Search API for active query: {query}", query=full_query)


DONE: search tool + plain text output
OUTPUT:
Finding a truly dark sky near Paris without a private car is challenging due to heavy light pollution. While the specialized "International Dark Sky Reserves" mentioned in my search (like the Morvan or Millevaches) are excellent, they are quite far from Paris and typically require a car to reach specific dark observation points once you arrive in the region by train.

However, if you are looking to escape the city center for better visibility using only public transport, here are three practical approaches:

1.  **Parc Naturel Régional du Gâtinais Français (Accessible via RER D / TER):**
    Located south of Paris, this area is often called the "Park of Bread and Roses." You can reach towns like **Maisse** or **Buno-Bonnevaux** via the RER D and a short TER connection. Once you are in these smaller villages, the light pollution drops significantly compared to the center of Paris. It is a popular spot for local amateur astronomers.

2.  **Pa

## 2a. Real search tool, JSON text output

This tests whether the model can use the search tool and emit JSON text when Pydantic AI does not force the final `output_type` tool.


In [18]:
from pydantic_ai.capabilities import WebSearch
from pydantic_ai.capabilities import Thinking


json_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    # capabilities=[SearchCapability()],
    capabilities=[WebSearch(), Thinking("high")],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Then return ONLY valid JSON text with keys: spots_found, summary, spots. "
        "spots must be an array of objects with name, latitude, longitude, source, description, accessibility, safety_assessment, and bortle_class. "
        "If search_spots returns an error string, return {\"spots_found\": 0, \"summary\": <error>, \"spots\": []}. "
        "Do not wrap the JSON in markdown. Do not invent spots that were not returned by the tool."
    ),
)

json_text_result = await run_with_timeout(
    "search tool + JSON text output",
    json_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("RAW OUTPUT:")
print(json_text_result.output)
print("\nPARSED JSON:")
pprint(json.loads(json_text_result.output))
print("\nUSAGE:")
print(json_text_result.usage)
print("\nMESSAGES:")
for message in json_text_result.all_messages():
    print(type(message).__name__, message)


/var/folders/lt/75xqxt7561731l87vw1h2_3w0000gn/T/ipykernel_11090/4026942807.py:9: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch(), Thinking("high")],


START: search tool + JSON text output
DONE: search tool + JSON text output
RAW OUTPUT:
{
"spots_found": 3,
"summary": "Three stargazing locations around Paris accessible via public transport were identified: Fontainebleau Forest, Rambouillet Forest, and the Limours Plateau. Each location is reachable by suburban train lines and serves as a significantly darker environment than central Paris, though they are still classified as suburban-rural transition zones.",
"spots": [
{
"name": "Fontainebleau Forest",
"latitude": 48.4239,
"longitude": 2.7011,
"source": "parissecret.com",
"description": "A vast forested area providing natural cover from nearby light pollution, often cited as a prime destination for observation in the Ile-de-France region.",
"accessibility": "Accessible via Transilien Line R from Gare de Lyon (Fontainebleau-Avon station).",
"safety_assessment": "Generally safe, but night-time navigation in forest clearings requires torches and adequate preparation.",
"bortle_class": 

## 3. Real search tool, structured Pydantic output

If steps 1 and 2 work but this fails, the problem is structured output/tool interaction. This cell forces a `SpotSearchReport` final result.


In [ ]:
structured_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    output_type=SpotSearchReport,
    capabilities=[WebSearch(), Thinking(effort=True)],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Use the tool result to fill SpotSearchReport. "
        "If search_spots returns an error string, return spots_found=0, spots=[], and put the error in summary. "
        "Do not invent spots that were not returned by the tool."
        "before returning rechack all fields for correctness and consistency, and if you find any issues, use the search tool again"
        "You can use search tool max 2 times for each spot"
    ),
)

structured_result = await run_with_timeout(
    "search tool + structured Pydantic output",
    structured_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("PYDANTIC OUTPUT:")
pprint(structured_result.output.model_dump())
print("\nJSON:")
print(structured_result.output.model_dump_json(indent=2))
print("\nUSAGE:")
print(structured_result.usage)
print("\nMESSAGES:")
for message in structured_result.all_messages():
    print(type(message).__name__, message)


/var/folders/lt/75xqxt7561731l87vw1h2_3w0000gn/T/ipykernel_11090/1373786464.py:5: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch(), Thinking(effort=True)],


START: search tool + structured Pydantic output
DONE: search tool + structured Pydantic output
PYDANTIC OUTPUT:
{'spots': [{'accessibility': 'Accessible via Transilien line R from Gare de '
                             'Lyon (approx. 40-50 min). A short walk or '
                             'shuttle is required to reach the forest edges '
                             'from the station.',
            'additional_info': None,
            'bortle_class': 4,
            'description': 'A vast, historic forest region southeast of Paris. '
                           'Its significant distance from the city center '
                           'provides a much darker sky than the urban core, '
                           'making it a popular choice for astronomy '
                           'enthusiasts seeking better visibility.',
            'latitude': 48.4286,
            'longitude': 2.7001,
            'name': 'Fontainebleau Forest',
            'safety_assessment': 'Generally safe, but r

: 